In [1]:
import open3d as o3d
import numpy as np
import copy
mesh=o3d.io.read_triangle_mesh("cup.obj")
pcd=mesh.sample_points_uniformly(number_of_points=20000)
pcd.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamKNN(knn=30))
pcd.orient_normals_consistent_tangent_plane(k=30)
fpfh=o3d.pipelines.registration.compute_fpfh_feature(
    pcd,o3d.geometry.KDTreeSearchParamKNN(knn=30))


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
P = np.asarray(pcd.points)
N = np.asarray(pcd.normals)
radial = P[:, :2] / np.linalg.norm(P[:, :2], axis=1, keepdims=True)
dot = (N[:, :2] * radial).sum(axis=1)
print((dot > 0).mean())

0.78915


In [3]:
P = np.asarray(pcd.points)
N = np.asarray(pcd.normals)
radial = P[:, :2] / np.linalg.norm(P[:, :2], axis=1, keepdims=True)
dot = (N[:, :2] * radial).sum(axis=1)
N[dot < 0] *= -1
pcd.normals = o3d.utility.Vector3dVector(N)
dot = (N[:, :2] * radial).sum(axis=1)
print((dot > 0).mean())

0.9977


In [4]:
pcd1=copy.deepcopy(pcd)
theta=np.radians(30)
R_gt=np.array([
    [1,0,0],
    [0,np.cos(theta),-np.sin(theta)],
    [0,np.sin(theta),np.cos(theta)]
    ])
t_gt=np.array([20.0,-10.0,15.0])
T_gt=np.eye(4)
T_gt[:3,:3]=R_gt
T_gt[:3,3]=t_gt
pcd1.transform(T_gt)

PointCloud with 20000 points.

In [5]:
def prep(pcd):
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(knn=30))
    pcd.orient_normals_consistent_tangent_plane(k=30)
    P=np.asarray(pcd.points);N=np.asarray(pcd.normals)
    radial=P[:,:2]/np.linalg.norm(P[:,:2],axis=1,keepdims=True)
    N[(N[:,:2]*radial).sum(1)<0]*=-1
    pcd.normals=o3d.utility.Vector3dVector(N)
    f=o3d.pipelines.registration.compute_fpfh_feature(pcd,o3d.geometry.KDTreeSearchParamKNN(knn=30))
    return pcd,f
pcd,f1=prep(pcd)
pcd1,f2=prep(pcd1)

In [6]:
pcd2 = mesh.sample_points_uniformly(number_of_points=20000)
P2 = np.asarray(pcd2.points) @ R_gt.T + t_gt
P2 += np.random.default_rng(0).normal(0, 0.3, P2.shape)
pcd2.points = o3d.utility.Vector3dVector(P2)
pcd2 = pcd2.voxel_down_sample(1.0)
pcd2,f3=prep(pcd2)

In [7]:
res=o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    pcd,pcd1,f1,f2,True,
    max_correspondence_distance=3.0,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    ransac_n=3,
    checkers=[o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
              o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(3.0)],
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(100000,0.999))

In [8]:
R_est = res.transformation[:3, :3]
t_est = res.transformation[:3, 3]
ang = np.degrees(np.arccos(np.clip((np.trace(R_est.T @ R_gt) - 1) / 2, -1, 1)))
print("角度误差", ang, "度")
print("平移误差", np.linalg.norm(t_est - t_gt), "mm")

角度误差 0.0 度
平移误差 1.5888218580782548e-14 mm


In [9]:
res1=o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    pcd,pcd2,f1,f3,True,
    max_correspondence_distance=3.0,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
    ransac_n=3,
    checkers=[o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
              o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(3.0)],
    criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(100000,0.999))

[Open3D WARNING] Too few correspondences (1868) after mutual filter, fall back to original correspondences.


In [10]:
R_est1 = res1.transformation[:3, :3]
t_est1 = res1.transformation[:3, 3]
ang = np.degrees(np.arccos(np.clip((np.trace(R_est1.T @ R_gt) - 1) / 2, -1, 1)))
print("角度误差", ang, "度")
print("平移误差", np.linalg.norm(t_est1 - t_gt), "mm")

角度误差 23.069686980669136 度
平移误差 1.3879722241334886 mm


In [11]:
pcd.paint_uniform_color([1, 0, 0])
pcd2.paint_uniform_color([0, 1, 0])
o3d.visualization.draw_geometries([pcd, pcd2])

In [12]:
res_icp = o3d.pipelines.registration.registration_icp(
    pcd, pcd2, 
    max_correspondence_distance=5.0, 
    init=res1.transformation,
    estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint()
)
R_est_icp = res_icp.transformation[:3, :3]
t_est_icp = res_icp.transformation[:3, 3]

In [13]:
ang_icp = np.degrees(np.arccos(np.clip((np.trace(R_est_icp.T @ R_gt) - 1) / 2, -1, 1)))

print("ICP精修后的角度误差", ang_icp, "度")
print("ICP精修后的平移误差", np.linalg.norm(t_est_icp - t_gt), "mm")

ICP精修后的角度误差 22.854521022180066 度
ICP精修后的平移误差 0.027559098559349456 mm


In [14]:
pcd_aligned = copy.deepcopy(pcd)
pcd_aligned.transform(res_icp.transformation)
pcd_aligned.paint_uniform_color([1, 0, 0])
o3d.visualization.draw_geometries([pcd_aligned, pcd2])

In [15]:
def init_form(delta_deg):
    a=np.radians(delta_deg)
    Rx=np.array([[1,0,0],
    [0,np.cos(a),-np.sin(a)],
    [0,np.sin(a),np.cos(a)]])
    T=np.eye(4)
    T[:3,:3]=Rx@R_gt
    T[:3,3]=t_gt
    return T

In [16]:
results = {}
for d in [0, 10, 30, 90, 180]:
    r = o3d.pipelines.registration.registration_icp(
        pcd, pcd2,
        max_correspondence_distance=5.0,
        init=init_form(d),
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane())
    results[d] = r

    R_e = r.transformation[:3, :3]
    t_e = r.transformation[:3, 3]
    ang = np.degrees(np.arccos(np.clip((np.trace(R_e.T @ R_gt) - 1) / 2, -1, 1)))
    print(f"初始偏差: {d:3} 度 | 最终角度误差: {ang:6.2f} 度 | "
          f"内点率: {r.fitness*100:6.2f}% | RMSE: {r.inlier_rmse:.4f} mm")
    print("-" * 70)
res_icp = results[0]
pcd_aligned = copy.deepcopy(pcd)
pcd_aligned.transform(res_icp.transformation)
evaluation = o3d.pipelines.registration.evaluate_registration(pcd_aligned, pcd2, 2.0)
print(f"内点率: {evaluation.fitness*100:.2f}%  RMSE: {evaluation.inlier_rmse:.4f} mm")

初始偏差:   0 度 | 最终角度误差:   0.57 度 | 内点率: 100.00% | RMSE: 0.9183 mm
----------------------------------------------------------------------
初始偏差:  10 度 | 最终角度误差:   8.39 度 | 内点率: 100.00% | RMSE: 0.9186 mm
----------------------------------------------------------------------
初始偏差:  30 度 | 最终角度误差:  41.72 度 | 内点率:  30.19% | RMSE: 2.6545 mm
----------------------------------------------------------------------
初始偏差:  90 度 | 最终角度误差: 153.97 度 | 内点率:   8.40% | RMSE: 2.5233 mm
----------------------------------------------------------------------
初始偏差: 180 度 | 最终角度误差: 180.00 度 | 内点率:   0.00% | RMSE: 0.0000 mm
----------------------------------------------------------------------
内点率: 99.59%  RMSE: 0.9100 mm


In [17]:
print(np.asarray(pcd2.points).mean(0))
print(P2.mean(0))

[ 19.98825277 -68.19417345 115.77746363]
[ 20.03321681 -68.27415273 115.7857479 ]


In [18]:
print(np.asarray(pcd_aligned.points).mean(0))
print(np.asarray(pcd2.points).mean(0))
print(res_icp.fitness, res_icp.inlier_rmse)

[ 20.05161864 -68.81714787 116.46882484]
[ 19.98825277 -68.19417345 115.77746363]
1.0 0.91830951778954


In [19]:
print(res1.fitness, res1.inlier_rmse)

0.7011 1.5685636665746892


In [ ]:
zv = np.array([0., 0., 1.])
R0 = results[0].transformation[:3, :3]
axis_err = np.degrees(np.arccos(np.clip(
    abs((R0 @ zv) @ (R_gt @ zv)), -1, 1)))
print("axis error", axis_err)
